In [1]:
import warnings
import numpy as np
import pandas as pd

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

warnings.filterwarnings("ignore")


# ============================================================
# 1. PARAMÈTRES
# ============================================================

INPUT_FILE = "dataset_finale_long.csv"
OUTPUT_FILE = "benchmark_arima_prophet_2024.csv"

COL_ZONE = "Zone géographique"
COL_YEAR = "Annee"
COL_POP = "Population"

TRAIN_START = 2004
TRAIN_END = 2023
TEST_YEAR = 2024

EPSILON = 1e-9


# ============================================================
# 2. CHARGEMENT DES DONNÉES
# ============================================================

df = pd.read_csv(
    INPUT_FILE,
    sep=";",
    encoding="utf-8-sig"
)

df.columns = df.columns.astype(str).str.strip()

# Gestion automatique si la colonne s'appelle "Année"
if "Année" in df.columns and "Annee" not in df.columns:
    df = df.rename(columns={"Année": "Annee"})

required_cols = [COL_ZONE, COL_YEAR, COL_POP]
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"Colonnes manquantes dans le dataset : {missing_cols}")

df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce").astype("Int64")
df[COL_POP] = pd.to_numeric(df[COL_POP], errors="coerce").astype("float64")

df = df.dropna(subset=[COL_ZONE, COL_YEAR, COL_POP]).copy()
df[COL_YEAR] = df[COL_YEAR].astype(int)

# Garder l'ordre original des zones comme dans le fichier source
df["ordre_original"] = pd.factorize(df[COL_ZONE])[0]

df = df.sort_values([COL_ZONE, COL_YEAR]).reset_index(drop=True)


# ============================================================
# 3. FONCTIONS DE MÉTRIQUES
# ============================================================

def compute_metrics(y_true, y_pred):
    """
    Calcule MAE, RMSE et MAPE pour une prédiction unique.
    """
    error = y_true - y_pred

    mae = abs(error)
    rmse = np.sqrt(error ** 2)

    if abs(y_true) < EPSILON:
        mape = np.nan
    else:
        mape = abs(error / y_true) * 100

    return mae, rmse, mape


# ============================================================
# 4. MODÈLE ARIMA
# ============================================================

def forecast_arima(train_series):
    """
    Teste plusieurs configurations ARIMA simples.
    Garde le modèle avec le meilleur AIC.
    Retourne uniquement la prédiction 2024.
    """

    candidate_orders = [
        (0, 1, 0),
        (1, 1, 0),
        (0, 1, 1),
        (1, 1, 1),
        (2, 1, 0),
        (0, 1, 2),
        (2, 1, 1),
        (1, 1, 2)
    ]

    best_aic = np.inf
    best_model = None

    for order in candidate_orders:
        try:
            model = ARIMA(
                train_series,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted_model = model.fit()

            if fitted_model.aic < best_aic:
                best_aic = fitted_model.aic
                best_model = fitted_model

        except Exception:
            continue

    if best_model is None:
        return np.nan

    forecast = best_model.forecast(steps=1)

    return float(forecast.iloc[0])


# ============================================================
# 5. MODÈLE PROPHET
# ============================================================

def forecast_prophet(train_df):
    """
    Entraîne Prophet sur 2004-2023 et prédit 2024.
    Données annuelles : pas de saisonnalité.
    """

    prophet_df = train_df[[COL_YEAR, COL_POP]].copy()

    prophet_df = prophet_df.rename(columns={
        COL_YEAR: "ds",
        COL_POP: "y"
    })

    prophet_df["ds"] = pd.to_datetime(
        prophet_df["ds"].astype(str) + "-01-01"
    )

    try:
        model = Prophet(
            yearly_seasonality=False,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode="additive"
        )

        model.fit(prophet_df)

        future = pd.DataFrame({
            "ds": [pd.to_datetime(f"{TEST_YEAR}-01-01")]
        })

        forecast = model.predict(future)

        pred_2024 = float(forecast["yhat"].iloc[0])

        return pred_2024

    except Exception:
        return np.nan


# ============================================================
# 6. BOUCLE PRINCIPALE PAR ZONE
# ============================================================

results = []

zones_order = (
    df[[COL_ZONE, "ordre_original"]]
    .drop_duplicates()
    .sort_values("ordre_original")[COL_ZONE]
    .tolist()
)

for zone in zones_order:
    df_zone = df[df[COL_ZONE] == zone].copy()
    df_zone = df_zone.sort_values(COL_YEAR)

    ordre_zone = int(df_zone["ordre_original"].iloc[0])

    train_df = df_zone[
        (df_zone[COL_YEAR] >= TRAIN_START) &
        (df_zone[COL_YEAR] <= TRAIN_END)
    ].copy()

    test_df = df_zone[df_zone[COL_YEAR] == TEST_YEAR].copy()

    if train_df.empty or test_df.empty:
        continue

    if len(train_df) < 5:
        continue

    y_true_2024 = float(test_df[COL_POP].iloc[0])

    train_series = train_df.set_index(COL_YEAR)[COL_POP].astype(float)

    # ---------------------------
    # ARIMA
    # ---------------------------

    pred_arima = forecast_arima(train_series)

    if np.isfinite(pred_arima):
        pred_arima = round(pred_arima)
        mae_arima, rmse_arima, mape_arima = compute_metrics(
            y_true_2024,
            pred_arima
        )
    else:
        mae_arima, rmse_arima, mape_arima = np.nan, np.nan, np.nan

    # ---------------------------
    # Prophet
    # ---------------------------

    pred_prophet = forecast_prophet(train_df)

    if np.isfinite(pred_prophet):
        pred_prophet = round(pred_prophet)
        mae_prophet, rmse_prophet, mape_prophet = compute_metrics(
            y_true_2024,
            pred_prophet
        )
    else:
        mae_prophet, rmse_prophet, mape_prophet = np.nan, np.nan, np.nan

    # ---------------------------
    # Sélection automatique
    # ---------------------------

    if np.isnan(mape_arima) and np.isnan(mape_prophet):
        best_model = "Aucun modèle valide"

    elif np.isnan(mape_arima):
        best_model = "Prophet"

    elif np.isnan(mape_prophet):
        best_model = "ARIMA"

    elif mape_arima <= mape_prophet:
        best_model = "ARIMA"

    else:
        best_model = "Prophet"

    results.append({
        "ordre_original": ordre_zone,
        "Zone géographique": zone,
        "Population réelle 2024": round(y_true_2024),

        "Prediction_ARIMA_2024": pred_arima,
        "MAE_ARIMA": mae_arima,
        "RMSE_ARIMA": rmse_arima,
        "MAPE_ARIMA_%": mape_arima,

        "Prediction_Prophet_2024": pred_prophet,
        "MAE_Prophet": mae_prophet,
        "RMSE_Prophet": rmse_prophet,
        "MAPE_Prophet_%": mape_prophet,

        "Modèle retenu": best_model
    })


# ============================================================
# 7. TABLEAU FINAL DE SYNTHÈSE
# ============================================================

benchmark_df = pd.DataFrame(results)

benchmark_df = benchmark_df.sort_values(
    by="ordre_original"
).reset_index(drop=True)

benchmark_df = benchmark_df.drop(columns=["ordre_original"])


# ============================================================
# 8. ARRONDIS DES VALEURS
# ============================================================

cols_entieres = [
    "Population réelle 2024",
    "Prediction_ARIMA_2024",
    "Prediction_Prophet_2024",
    "MAE_ARIMA",
    "RMSE_ARIMA",
    "MAE_Prophet",
    "RMSE_Prophet"
]

for col in cols_entieres:
    benchmark_df[col] = benchmark_df[col].round(0).astype("Int64")

cols_mape = [
    "MAPE_ARIMA_%",
    "MAPE_Prophet_%"
]

for col in cols_mape:
    benchmark_df[col] = benchmark_df[col].round(3)


# ============================================================
# 9. EXPORT
# ============================================================

benchmark_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)


# ============================================================
# 10. RÉSUMÉ
# ============================================================

print("Benchmark terminé avec succès.")
print(f"Nombre de zones évaluées : {len(benchmark_df)}")
print(f"Fichier généré : {OUTPUT_FILE}")

print("\nRépartition des modèles retenus :")
print(benchmark_df["Modèle retenu"].value_counts())

print("\nAperçu du tableau final :")
print(benchmark_df.head())

c:\Users\LENOVO\.conda\envs\pfe_hcp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.
16:50:52 - cmdstanpy - INFO - Chain [1] start processing
16:50:52 - cmdstanpy - INFO - Chain [1] done processing
16:50:52 - cmdstanpy - INFO - Chain [1] start processing
16:50:53 - cmdstanpy - INFO - Chain [1] done processing
16:50:53 - cmdstanpy - INFO - Chain [1] start processing
16:50:53 - cmdstanpy - INFO - Chain [1] done processing
16:50:54 - cmdstanpy - INFO - Chain [1] start processing
16:50:54 - cmdstanpy - INFO - Chain [1] done processing
16:50:55 - cmdstanpy - INFO - Chain [1] start processing
16:50:55 - cmdstanpy - INFO - Chain [1] done processing
16:50:55 - cmdstanpy - INFO - Chain [1] start processing
16:50:56 - cmdstanpy - INFO - Chain [1] done processing
16:50

Benchmark terminé avec succès.
Nombre de zones évaluées : 177
Fichier généré : benchmark_arima_prophet_2024.csv

Répartition des modèles retenus :
Modèle retenu
ARIMA      124
Prophet     53
Name: count, dtype: int64

Aperçu du tableau final :
           Zone géographique  Population réelle 2024  Prediction_ARIMA_2024  \
0                   National                36828330               36826091   
1  Tanger-Tétouan-Al Hoceima                 4030222                4031066   
2          Al Hoceima (Prov)                  371527                 371609   
3                 Al Hoceima                   50225                  50251   
4               Bni Bouayach                   20013                  20032   

   MAE_ARIMA  RMSE_ARIMA  MAPE_ARIMA_%  Prediction_Prophet_2024  MAE_Prophet  \
0       2239        2239         0.006                 36844139        15809   
1        844         844         0.021                  4032535         2313   
2         82          82         0.022   

In [2]:
import warnings
import numpy as np
import pandas as pd

from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

warnings.filterwarnings("ignore")


# ============================================================
# 1. PARAMÈTRES
# ============================================================

WIDE_FILE = "dataset_finale.csv"
DATA_FILE = "dataset_finale_long.csv"
BENCHMARK_FILE = "benchmark_arima_prophet_2024.csv"

OUTPUT_WIDE_FILE = "projections_2025_2040_wide.csv"
OUTPUT_LONG_FILE = "projections_2025_2040_long.csv"

COL_ZONE = "Zone géographique"
COL_YEAR = "Annee"
COL_POP = "Population"
COL_MODEL = "Modèle retenu"

HIST_START = 2004
HIST_END = 2024

FORECAST_START = 2025
FORECAST_END = 2040
FORECAST_YEARS = list(range(FORECAST_START, FORECAST_END + 1))


# ============================================================
# 2. CHARGEMENT DES DONNÉES
# ============================================================

df = pd.read_csv(
    DATA_FILE,
    sep=";",
    encoding="utf-8-sig"
)

benchmark = pd.read_csv(
    BENCHMARK_FILE,
    sep=";",
    encoding="utf-8-sig"
)

df_wide_order = pd.read_csv(
    WIDE_FILE,
    sep=";",
    encoding="utf-8-sig"
)

df.columns = df.columns.astype(str).str.strip()
benchmark.columns = benchmark.columns.astype(str).str.strip()
df_wide_order.columns = df_wide_order.columns.astype(str).str.strip()

# Gestion automatique si la colonne s'appelle "Année"
if "Année" in df.columns and "Annee" not in df.columns:
    df = df.rename(columns={"Année": "Annee"})

required_data_cols = [COL_ZONE, COL_YEAR, COL_POP]
required_benchmark_cols = [COL_ZONE, COL_MODEL]
required_wide_cols = [COL_ZONE]

missing_data = [c for c in required_data_cols if c not in df.columns]
missing_benchmark = [c for c in required_benchmark_cols if c not in benchmark.columns]
missing_wide = [c for c in required_wide_cols if c not in df_wide_order.columns]

if missing_data:
    raise ValueError(f"Colonnes manquantes dans {DATA_FILE} : {missing_data}")

if missing_benchmark:
    raise ValueError(f"Colonnes manquantes dans {BENCHMARK_FILE} : {missing_benchmark}")

if missing_wide:
    raise ValueError(f"Colonnes manquantes dans {WIDE_FILE} : {missing_wide}")


# ============================================================
# 3. NETTOYAGE DES DONNÉES
# ============================================================

df[COL_YEAR] = pd.to_numeric(df[COL_YEAR], errors="coerce").astype("Int64")
df[COL_POP] = pd.to_numeric(df[COL_POP], errors="coerce").astype("float64")

df = df.dropna(subset=[COL_ZONE, COL_YEAR, COL_POP]).copy()
df[COL_YEAR] = df[COL_YEAR].astype(int)

df[COL_ZONE] = df[COL_ZONE].astype(str).str.strip()
benchmark[COL_ZONE] = benchmark[COL_ZONE].astype(str).str.strip()
benchmark[COL_MODEL] = benchmark[COL_MODEL].astype(str).str.strip()
df_wide_order[COL_ZONE] = df_wide_order[COL_ZONE].astype(str).str.strip()

df = df.sort_values([COL_ZONE, COL_YEAR]).reset_index(drop=True)


# ============================================================
# 4. ORDRE ORIGINAL DES ZONES DEPUIS dataset_finale.csv
# ============================================================

ordre_zones = df_wide_order[[COL_ZONE]].copy()
ordre_zones = ordre_zones.drop_duplicates().reset_index(drop=True)
ordre_zones["ordre_original"] = range(len(ordre_zones))


# ============================================================
# 5. FONCTION ARIMA
# ============================================================

def forecast_arima_full(train_series, steps):
    """
    Entraîne ARIMA sur toute la série 2004-2024.
    Sélection du meilleur ordre par AIC parmi plusieurs modèles simples.
    """

    candidate_orders = [
        (0, 1, 0),
        (1, 1, 0),
        (0, 1, 1),
        (1, 1, 1),
        (2, 1, 0),
        (0, 1, 2),
        (2, 1, 1),
        (1, 1, 2)
    ]

    best_aic = np.inf
    best_model = None

    for order in candidate_orders:
        try:
            model = ARIMA(
                train_series,
                order=order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )

            fitted = model.fit()

            if fitted.aic < best_aic:
                best_aic = fitted.aic
                best_model = fitted

        except Exception:
            continue

    if best_model is None:
        return [np.nan] * steps

    forecast = best_model.forecast(steps=steps)

    return forecast.values.tolist()


# ============================================================
# 6. FONCTION PROPHET
# ============================================================

def forecast_prophet_full(train_df, future_years):
    """
    Entraîne Prophet sur toute la série 2004-2024.
    Prévoit de 2025 jusqu'à 2040.
    Données annuelles : pas de saisonnalité.
    """

    prophet_df = train_df[[COL_YEAR, COL_POP]].copy()

    prophet_df = prophet_df.rename(columns={
        COL_YEAR: "ds",
        COL_POP: "y"
    })

    prophet_df["ds"] = pd.to_datetime(
        prophet_df["ds"].astype(str) + "-01-01"
    )

    try:
        model = Prophet(
            yearly_seasonality=False,
            weekly_seasonality=False,
            daily_seasonality=False,
            seasonality_mode="additive"
        )

        model.fit(prophet_df)

        future = pd.DataFrame({
            "ds": [pd.to_datetime(f"{year}-01-01") for year in future_years]
        })

        forecast = model.predict(future)

        return forecast["yhat"].values.tolist()

    except Exception:
        return [np.nan] * len(future_years)


# ============================================================
# 7. PRÉPARATION DU TABLEAU DES MODÈLES RETENUS
# ============================================================

benchmark_models = benchmark[[COL_ZONE, COL_MODEL]].copy()

df_model = ordre_zones.merge(
    benchmark_models,
    on=COL_ZONE,
    how="left"
)

missing_model = df_model[df_model[COL_MODEL].isna()].copy()

if not missing_model.empty:
    print("Attention : certaines zones n'ont pas de modèle retenu dans le benchmark.")
    print("Elles seront ignorées.")
    print(missing_model[[COL_ZONE]].head(20))


# ============================================================
# 8. BOUCLE DE PROJECTION 2025-2040
# ============================================================

projection_rows = []

for _, row in df_model.iterrows():
    zone = row[COL_ZONE]
    modele_retenu = row[COL_MODEL]

    if pd.isna(modele_retenu):
        continue

    df_zone = df[
        (df[COL_ZONE] == zone) &
        (df[COL_YEAR] >= HIST_START) &
        (df[COL_YEAR] <= HIST_END)
    ].copy()

    df_zone = df_zone.sort_values(COL_YEAR)

    if len(df_zone) < 5:
        continue

    train_series = df_zone.set_index(COL_YEAR)[COL_POP].astype(float)

    modele_retenu_clean = str(modele_retenu).strip().lower()

    if modele_retenu_clean == "arima":
        preds = forecast_arima_full(
            train_series=train_series,
            steps=len(FORECAST_YEARS)
        )

    elif modele_retenu_clean == "prophet":
        preds = forecast_prophet_full(
            train_df=df_zone,
            future_years=FORECAST_YEARS
        )

    else:
        preds = [np.nan] * len(FORECAST_YEARS)

    # Sécurité : population non négative
    preds = [
        max(0, pred) if np.isfinite(pred) else np.nan
        for pred in preds
    ]

    # Arrondi final : population = individus
    preds = [
        round(pred) if np.isfinite(pred) else np.nan
        for pred in preds
    ]

    result = {
        "ordre_original": row["ordre_original"],
        COL_ZONE: zone,
        "Modèle utilisé": modele_retenu
    }

    for year, pred in zip(FORECAST_YEARS, preds):
        result[str(year)] = pred

    projection_rows.append(result)


# ============================================================
# 9. CRÉATION FORMAT WIDE
# ============================================================

projections_wide = pd.DataFrame(projection_rows)

projections_wide = projections_wide.sort_values(
    by="ordre_original"
).reset_index(drop=True)

projections_wide = projections_wide.drop(columns=["ordre_original"])

year_cols = [str(y) for y in FORECAST_YEARS]

for col in year_cols:
    projections_wide[col] = projections_wide[col].round(0).astype("Int64")


# ============================================================
# 10. CRÉATION FORMAT LONG
# ============================================================

projections_long = projections_wide.melt(
    id_vars=[COL_ZONE, "Modèle utilisé"],
    value_vars=year_cols,
    var_name="Annee",
    value_name="Population projetée"
)

projections_long["Annee"] = projections_long["Annee"].astype(int)
projections_long["Population projetée"] = (
    projections_long["Population projetée"]
    .round(0)
    .astype("Int64")
)

# Garder l'ordre original : zone puis année
projections_long = projections_long.sort_values(
    by=[COL_ZONE, "Annee"]
).reset_index(drop=True)

# Si tu veux garder l'ordre original exact du fichier source dans le long aussi :
ordre_map = {
    zone: i
    for i, zone in enumerate(projections_wide[COL_ZONE].tolist())
}

projections_long["ordre_original"] = projections_long[COL_ZONE].map(ordre_map)

projections_long = projections_long.sort_values(
    by=["ordre_original", "Annee"]
).reset_index(drop=True)

projections_long = projections_long.drop(columns=["ordre_original"])


# ============================================================
# 11. EXPORT DES DEUX FORMATS
# ============================================================

projections_wide.to_csv(
    OUTPUT_WIDE_FILE,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)

projections_long.to_csv(
    OUTPUT_LONG_FILE,
    index=False,
    encoding="utf-8-sig",
    sep=";"
)


# ============================================================
# 12. RÉSUMÉ
# ============================================================

print("Projection terminée avec succès.")
print(f"Horizon de projection : {FORECAST_START}-{FORECAST_END}")

print(f"\nNombre de zones projetées : {len(projections_wide)}")

print("\nFichiers générés :")
print(f"- {OUTPUT_WIDE_FILE}")
print(f"- {OUTPUT_LONG_FILE}")

print("\nRépartition des modèles utilisés :")
print(projections_wide["Modèle utilisé"].value_counts())

print("\nAperçu format wide :")
print(projections_wide.head())

print("\nAperçu format long :")
print(projections_long.head(20))

14:48:04 - cmdstanpy - INFO - Chain [1] start processing
14:48:06 - cmdstanpy - INFO - Chain [1] done processing
14:48:10 - cmdstanpy - INFO - Chain [1] start processing
14:48:10 - cmdstanpy - INFO - Chain [1] done processing
14:48:11 - cmdstanpy - INFO - Chain [1] start processing
14:48:11 - cmdstanpy - INFO - Chain [1] done processing
14:48:14 - cmdstanpy - INFO - Chain [1] start processing
14:48:14 - cmdstanpy - INFO - Chain [1] done processing
14:48:14 - cmdstanpy - INFO - Chain [1] start processing
14:48:15 - cmdstanpy - INFO - Chain [1] done processing
14:48:19 - cmdstanpy - INFO - Chain [1] start processing
14:48:20 - cmdstanpy - INFO - Chain [1] done processing
14:48:22 - cmdstanpy - INFO - Chain [1] start processing
14:48:22 - cmdstanpy - INFO - Chain [1] done processing
14:48:24 - cmdstanpy - INFO - Chain [1] start processing
14:48:25 - cmdstanpy - INFO - Chain [1] done processing
14:48:25 - cmdstanpy - INFO - Chain [1] start processing
14:48:26 - cmdstanpy - INFO - Chain [1]

Projection terminée avec succès.
Horizon de projection : 2025-2040

Nombre de zones projetées : 177

Fichiers générés :
- projections_2025_2040_wide.csv
- projections_2025_2040_long.csv

Répartition des modèles utilisés :
Modèle utilisé
ARIMA      124
Prophet     53
Name: count, dtype: int64

Aperçu format wide :
           Zone géographique Modèle utilisé      2025      2026      2027  \
0                   National          ARIMA  37114597  37394683  37668659   
1  Tanger-Tétouan-Al Hoceima          ARIMA   4076436   4122518   4168497   
2          Al Hoceima (Prov)          ARIMA    368240    364908    361532   
3                 Al Hoceima          ARIMA     49362     48472     47554   
4               Bni Bouayach          ARIMA     20129     20241     20349   

       2028      2029      2030      2031      2032      2033      2034  \
0  37936658  38198813  38455248  38706091  38951461  39191480  39426263   
1   4214371   4260143   4305812   4351377   4396840   4442201   4487460 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
# Chargement du fichier de benchmark
df_bench = pd.read_csv("benchmark_arima_prophet_2024.csv", sep=";")
# Création des colonnes pour les performances du modèle final
df_bench['MAE_Retenu'] = np.where(df_bench['Modèle retenu'] == 'ARIMA', df_bench['MAE_ARIMA'], df_bench['MAE_Prophet'])

df_bench['RMSE_Retenu'] = np.where(df_bench['Modèle retenu'] == 'ARIMA', df_bench['RMSE_ARIMA'], df_bench['RMSE_Prophet'])

df_bench['MAPE_Retenu_%'] = np.where(df_bench['Modèle retenu'] == 'ARIMA', df_bench['MAPE_ARIMA_%'], df_bench['MAPE_Prophet_%'])
# Construction du DataFrame avec les 6 lignes exactes
tab_3_3 = pd.DataFrame({
    'Indicateur': [
        'MAE Moyen (Erreur Absolue)', 
        'MAE Médian', 
        'RMSE Moyen (Erreur Quadratique)', 
        'RMSE Médian', 
        'MAPE Moyen (%)', 
        'MAPE Médian (%)'
    ],
    'ARIMA': [
        f"{df_bench['MAE_ARIMA'].mean():.2f}",
        f"{df_bench['MAE_ARIMA'].median():.2f}",
        f"{df_bench['RMSE_ARIMA'].mean():.2f}",
        f"{df_bench['RMSE_ARIMA'].median():.2f}",
        f"{df_bench['MAPE_ARIMA_%'].mean() * 100:.2f} %",
        f"{df_bench['MAPE_ARIMA_%'].median() * 100:.2f} %"
    ],
    'Prophet': [
        f"{df_bench['MAE_Prophet'].mean():.2f}",
        f"{df_bench['MAE_Prophet'].median():.2f}",
        f"{df_bench['RMSE_Prophet'].mean():.2f}",
        f"{df_bench['RMSE_Prophet'].median():.2f}",
        f"{df_bench['MAPE_Prophet_%'].mean() * 100:.2f} %",
        f"{df_bench['MAPE_Prophet_%'].median() * 100:.2f} %"
    ],
    'Modèle Retenu': [
        f"{df_bench['MAE_Retenu'].mean():.2f}",
        f"{df_bench['MAE_Retenu'].median():.2f}",
        f"{df_bench['RMSE_Retenu'].mean():.2f}",
        f"{df_bench['RMSE_Retenu'].median():.2f}",
        f"{df_bench['MAPE_Retenu_%'].mean() * 100:.2f} %",
        f"{df_bench['MAPE_Retenu_%'].median() * 100:.2f} %"
    ]
})
# Création du dossier si nécessaire
os.makedirs("Tableaux", exist_ok=True)

# 1. Sauvegarde en CSV
tab_3_3.to_csv("Tableaux/Tableau_3.3_Synthese_erreurs.csv", index=False, sep=";")

# 2. Création de l'image PNG stylisée
fig, ax = plt.subplots(figsize=(10, 3.5)) # Taille ajustée pour 6 lignes
ax.axis('off')

# Dessin du tableau
table = ax.table(cellText=tab_3_3.values, colLabels=tab_3_3.columns, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)

# Application des couleurs (Bleu)
BLUE_MAIN = "#0077B6"
BLUE_BG = "#CAF0F8"
BLUE_DARK = "#03045E"

for (row, col), cell in table.get_celld().items():
    if row == 0: # En-tête
        cell.set_facecolor(BLUE_MAIN)
        cell.set_text_props(color='white', weight='bold')
    else: # Lignes de données
        cell.set_facecolor(BLUE_BG if row % 2 == 0 else 'white')
        cell.set_text_props(color=BLUE_DARK)
        
        # Mettre la colonne "Indicateur" alignée à gauche pour plus de propreté
        if col == 0:
            cell.set_text_props(ha='left')
            
    cell.set_edgecolor('white')

plt.title("Tableau 3.3 — Synthèse globale des erreurs", fontsize=14, fontweight='bold', color=BLUE_DARK, pad=20)
plt.tight_layout()
plt.savefig("Tableaux/Tableau_3.3_Synthese.png", dpi=300, bbox_inches='tight')
plt.close()

print("Le tableau 3.3 a été généré avec succès !")

Le tableau 3.3 a été généré avec succès !


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# 1. Chargement des données
df_bench = pd.read_csv("benchmark_arima_prophet_2024.csv", sep=";")

# 2. Récupération des valeurs du Modèle Retenu (MAPE et Prédiction estimée)
df_bench['MAPE_Retenu_%'] = np.where(
    df_bench['Modèle retenu'] == 'ARIMA', 
    df_bench['MAPE_ARIMA_%'], 
    df_bench['MAPE_Prophet_%']
)

# On récupère la population estimée par le modèle gagnant
df_bench['Population estimée 2024'] = np.where(
    df_bench['Modèle retenu'] == 'ARIMA', 
    df_bench['Prediction_ARIMA_2024'], 
    df_bench['Prediction_Prophet_2024']
).astype(int) # On s'assure d'avoir des nombres entiers

# 3. Extraction du Top 10 des pires erreurs
top_10 = df_bench.nlargest(10, 'MAPE_Retenu_%').copy()

# 4. Sélection des 5 colonnes exactes dans le bon ordre
tab_3_4 = top_10[[
    'Zone géographique', 
    'Population réelle 2024', 
    'Population estimée 2024', 
    'Modèle retenu', 
    'MAPE_Retenu_%'
]].copy()

# Formatage de la colonne MAPE (* 100, 2 décimales, avec le symbole %)
tab_3_4.rename(columns={'MAPE_Retenu_%': 'MAPE (%)'}, inplace=True)
tab_3_4['MAPE (%)'] = (tab_3_4['MAPE (%)'] * 100).round(2).astype(str) + ' %'

# 5. Génération UNIQUEMENT de l'image PNG (sans export CSV)
os.makedirs("Tableaux", exist_ok=True)
fig, ax = plt.subplots(figsize=(10, 4.5)) # On élargit un peu pour faire tenir les 5 colonnes proprement
ax.axis('off')

# Création du tableau visuel
table = ax.table(cellText=tab_3_4.values, colLabels=tab_3_4.columns, loc='center', cellLoc='center')

# Application du style (Camaïeu de bleus)
BLUE_MAIN = "#0077B6"
BLUE_BG = "#CAF0F8"
BLUE_DARK = "#03045E"

table.auto_set_font_size(False)
table.set_fontsize(10) # Police ajustée pour la largeur
table.scale(1, 1.8)

for (row, col), cell in table.get_celld().items():
    if row == 0: # En-tête
        cell.set_facecolor(BLUE_MAIN)
        cell.set_text_props(color='white', weight='bold')
    else: # Lignes
        cell.set_facecolor(BLUE_BG if row % 2 == 0 else 'white')
        cell.set_text_props(color=BLUE_DARK)
        
        # Aligner le nom des villes à gauche pour plus de clarté
        if col == 0:
            cell.set_text_props(ha='left')
            
    cell.set_edgecolor('white')

plt.title("Tableau 3.4 — Top 10 des erreurs de prévision (Backtesting 2024)", 
          fontsize=14, fontweight='bold', color=BLUE_DARK, pad=15)
plt.tight_layout()

# Sauvegarde finale en Image uniquement
plt.savefig("Tableaux/Tableau_3.4_Top10_Erreurs.png", dpi=300, bbox_inches='tight')
plt.close()

print("L'image du Tableau 3.4 a été générée avec succès  !")

L'image du Tableau 3.4 a été générée avec succès  !


In [2]:
!pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ==========================================
# 0. CONFIGURATION ET PRÉPARATION
# ==========================================
os.makedirs("Figures", exist_ok=True)

# Charte graphique (Camaïeu de bleus)
BLUE_DARK = "#03045E"   
BLUE_MAIN = "#0077B6"   
BLUE_LIGHT = "#00B4D8"  
BLUE_PALE = "#90E0EF"   
BLUE_BG = "#CAF0F8"     

plt.rcParams.update({
    'font.family': 'sans-serif',
    'axes.facecolor': '#F8F9FA',
    'axes.edgecolor': BLUE_MAIN,
    'axes.labelcolor': BLUE_DARK,
    'text.color': BLUE_DARK,
    'xtick.color': BLUE_MAIN,
    'ytick.color': BLUE_MAIN,
    'grid.color': '#E9ECEF',
    'figure.facecolor': 'white'
})

# Chargement des fichiers
df_hist = pd.read_csv("dataset_finale.csv", sep=";")
df_bench = pd.read_csv("benchmark_arima_prophet_2024.csv", sep=";")
df_proj = pd.read_csv("projections_2025_2040_wide.csv", sep=";") # Nouveau fichier 2040

# MAPE du modèle finalement retenu
df_bench['MAPE_Retenu_%'] = np.where(
    df_bench['Modèle retenu'] == 'ARIMA', 
    df_bench['MAPE_ARIMA_%'], 
    df_bench['MAPE_Prophet_%']
)


# ==========================================
# 1. GÉNÉRATION DES FIGURES
# ==========================================
print("Génération des figures...")

# 1. Protocole de backtesting
plt.figure(figsize=(10, 4))
plt.axis('off')
boxes = [
    ("1. Données Historiques\n(2004-2023)", 0.1, 0.5),
    ("2. Entraînement\nARIMA & Prophet", 0.4, 0.5),
    ("3. Backtest vs 2024\n(Calcul MAPE)", 0.7, 0.7),
    ("4. Projection Finale\n(2025-2040)", 0.7, 0.3) # Modifié 2040
]
for text, x, y in boxes:
    plt.text(x, y, text, ha='center', va='center', fontsize=11, fontweight='bold',
             bbox=dict(boxstyle="round,pad=1", facecolor=BLUE_BG, edgecolor=BLUE_MAIN, lw=2))
plt.annotate("", xy=(0.25, 0.5), xytext=(0.55, 0.5), arrowprops=dict(arrowstyle="<-", color=BLUE_DARK, lw=2))
plt.annotate("", xy=(0.55, 0.65), xytext=(0.4, 0.55), arrowprops=dict(arrowstyle="<-", color=BLUE_DARK, lw=2))
plt.annotate("", xy=(0.55, 0.35), xytext=(0.4, 0.45), arrowprops=dict(arrowstyle="<-", color=BLUE_DARK, lw=2))
plt.title("Schéma du protocole de backtesting et de projection", fontsize=14, fontweight='bold', color=BLUE_DARK)
plt.savefig("Figures/Protocole_Backtesting.png", dpi=300, bbox_inches='tight')
plt.close()

# 2. Répartition ARIMA vs Prophet (Avec affichage des valeurs)
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df_bench, x='Modèle retenu', hue='Modèle retenu', palette=[BLUE_MAIN, BLUE_LIGHT], edgecolor=BLUE_DARK, legend=False)
plt.title("Répartition des modèles prédictifs retenus", fontsize=14, fontweight='bold', pad=15)
plt.ylabel("Nombre de zones géographiques")

# Boucle pour ajouter les chiffres exacts au-dessus des barres
for container in ax.containers:
    ax.bar_label(container, fontsize=12, fontweight='bold', color=BLUE_DARK, padding=5)

plt.savefig("Figures/Repartition_Modeles.png", dpi=300, bbox_inches='tight')
plt.close()

# 3. Distribution du MAPE
plt.figure(figsize=(8, 5))
sns.histplot(df_bench['MAPE_Retenu_%'] * 100, bins=20, kde=True, color=BLUE_MAIN, edgecolor=BLUE_DARK)
plt.title("Distribution de l'erreur absolue moyenne en pourcentage (MAPE)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("MAPE (%)")
plt.ylabel("Fréquence")
plt.savefig("Figures/Distribution_MAPE.png", dpi=300, bbox_inches='tight')
plt.close()

# 4. Top 10 erreurs
top10_err = df_bench.nlargest(10, 'MAPE_Retenu_%')
plt.figure(figsize=(10, 6))
sns.barplot(data=top10_err, x='MAPE_Retenu_%', y='Zone géographique', hue='Zone géographique', palette='Blues_r', edgecolor=BLUE_DARK, legend=False)
plt.title("Top 10 des zones présentant les erreurs de prévision les plus fortes", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("MAPE")
plt.savefig("Figures/Top10_Erreurs.png", dpi=300, bbox_inches='tight')
plt.close()

# Fonction pour tracer une évolution modifiée pour l'horizon 2040
def tracer_evolution(zone, titre, filename):
    h = df_hist[df_hist['Zone géographique'] == zone].iloc[:, 1:].values.flatten()
    p = df_proj[df_proj['Zone géographique'] == zone].iloc[:, 2:].values.flatten()
    
    annees_h = np.arange(2004, 2025)
    annees_p = np.arange(2025, 2041) # Modifié 2040
    
    plt.figure(figsize=(10, 5))
    plt.plot(annees_h, h, marker='o', color=BLUE_DARK, label='Historique', lw=2)
    plt.plot([2024, 2025], [h[-1], p[0]], 'k--', alpha=0.5)
    plt.plot(annees_p, p, marker='s', linestyle='--', color=BLUE_LIGHT, label='Projection', lw=2)
    
    # Ombre de prévision étendue jusqu'à 2040
    plt.axvspan(2024.5, 2040, color=BLUE_PALE, alpha=0.2)
    
    plt.title(titre, fontsize=14, fontweight='bold', color=BLUE_DARK)
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.savefig(f"Figures/{filename}.png", dpi=300, bbox_inches='tight')
    plt.close()

# 5. Évolution globale
tracer_evolution("Tanger-Tétouan-Al Hoceima", "Évolution démographique globale de la région (2004–2040)", "Evolution_Globale")

# 6. Top 10 population 2040
df_proj_clean = df_proj[~df_proj['Zone géographique'].isin(['National', 'Tanger-Tétouan-Al Hoceima'])]
top10_pop = df_proj_clean.nlargest(10, '2040') # Pointe vers 2040
plt.figure(figsize=(10, 6))
sns.barplot(data=top10_pop, x='2040', y='Zone géographique', hue='Zone géographique', palette='Blues_r', edgecolor=BLUE_DARK, legend=False)
plt.title("Top 10 des zones géographiques les plus peuplées en 2040", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Population (2040)")
plt.savefig("Figures/Top10_Pop_2040.png", dpi=300, bbox_inches='tight')
plt.close()

# 7. Top 10 croissance
df_croissance = pd.merge(df_hist[['Zone géographique', '2024']], df_proj_clean[['Zone géographique', '2040']], on='Zone géographique')
# Calcul par rapport à 2040
df_croissance['Croissance_%'] = ((df_croissance['2040'] - df_croissance['2024']) / df_croissance['2024']) * 100
top10_croissance = df_croissance.nlargest(10, 'Croissance_%')

plt.figure(figsize=(10, 6))
sns.barplot(data=top10_croissance, x='Croissance_%', y='Zone géographique', hue='Zone géographique', palette='Blues_r', edgecolor=BLUE_DARK, legend=False)
plt.title("Top 10 des taux de croissance démographique prévus (2024–2040)", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Croissance (%)")
plt.savefig("Figures/Top10_Croissance_2040.png", dpi=300, bbox_inches='tight')
plt.close()

# 8. Exemple projection ARIMA
tracer_evolution("Saddina", "Exemple d'une projection modélisée par ARIMA (Saddina)", "Exemple_ARIMA")

# 9. Exemple projection Prophet
tracer_evolution("Azla", "Exemple d'une projection modélisée par Prophet (Azla)", "Exemple_Prophet")

print("Terminé ! Toutes les figures ont été crées avec succès.")

Génération des figures...
Terminé ! Toutes les figures ont été crées avec succès.
